In [1]:
import torch 
print(f"GPU: {torch.cuda.get_device_name(0)}") 

GPU: Tesla T4


In [3]:
import pandas as pd 
import numpy as np 
import torch 
import csv 
import os 
from transformers import AutoTokenizer, AutoModel 
from torch.optim import AdamW 
from torch.utils.data import Dataset, DataLoader 
from transformers import get_linear_schedule_with_warmup 
from scipy.stats import spearmanr 
from sklearn.metrics import mean_squared_error 
DATA_ROOT   = '/kaggle/input/datasets/resegomorei/semrel2024-raw/raw' 
RESULTS_LOG = '/kaggle/working/results_log.csv' 
CKPT_DIR    = '/kaggle/working/checkpoints' 
MODEL_NAME  = 'castorini/afriberta_large' 
MAX_LEN     = 128 
BATCH_SIZE  = 32  # AfriBERTa is smaller, can use larger batch 
EPOCHS      = 10 
LR          = 2e-5 
PATIENCE    = 3 
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
 
os.makedirs(CKPT_DIR, exist_ok=True) 
print(f"Device: {DEVICE}") 
print(f"Model:  {MODEL_NAME}") 

Device: cuda
Model:  castorini/afriberta_large


In [4]:
class SemRelDataset(Dataset): 
    def __init__(self, df, tokenizer, max_len): 
        self.df        = df.reset_index(drop=True) 
        self.tokenizer = tokenizer 
        self.max_len   = max_len 
 
    def __len__(self): 
        return len(self.df) 
 
    def __getitem__(self, idx): 
        row = self.df.iloc[idx] 
        enc = self.tokenizer( 
            str(row['sentence1']), 
            str(row['sentence2']), 
            truncation=True, 
            max_length=self.max_len, 
            padding='max_length', 
            return_tensors='pt' 
        ) 
        return { 
            'input_ids':      enc['input_ids'].squeeze(), 
            'attention_mask': enc['attention_mask'].squeeze(), 
            'label':          torch.tensor(row['label'], dtype=torch.float) 
        } 
 
print("SemRelDataset defined.") 

SemRelDataset defined.


In [5]:
class AfriBERTaModel(torch.nn.Module): 
    def __init__(self, model_name): 
        super().__init__() 
        self.encoder   = AutoModel.from_pretrained(model_name) 
        self.regressor = torch.nn.Linear(self.encoder.config.hidden_size, 1) 
        self.dropout   = torch.nn.Dropout(0.1) 
 
    def forward(self, input_ids, attention_mask): 
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask) 
        cls = outputs.last_hidden_state[:, 0, :] 
        return self.regressor(self.dropout(cls)).squeeze(-1) 
 
 
def evaluate(model, loader): 
    model.eval() 
    preds, labels = [], [] 
    with torch.no_grad(): 
        for batch in loader: 
            ids  = batch['input_ids'].to(DEVICE) 
            mask = batch['attention_mask'].to(DEVICE) 
            out  = model(ids, mask).cpu().numpy() 
            preds.extend(out) 
            labels.extend(batch['label'].numpy()) 
    rho, _ = spearmanr(preds, labels) 
    mse    = mean_squared_error(labels, preds) 
    return rho, mse, preds 
 
 
def train(lang_code, lang_name, tokenizer): 
    print(f"\n{'='*60}") 
    print(f"  Training AfriBERTa on {lang_name}") 
    print(f"{'='*60}") 
 
    train_df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/train.csv') 
    dev_df   = pd.read_csv(f'{DATA_ROOT}/{lang_code}/dev.csv') 
    test_df  = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv') 
 
    train_loader = DataLoader(SemRelDataset(train_df, tokenizer, MAX_LEN), 
                              batch_size=BATCH_SIZE, shuffle=True) 
    dev_loader   = DataLoader(SemRelDataset(dev_df,   tokenizer, MAX_LEN), 
                              batch_size=BATCH_SIZE) 
    test_loader  = DataLoader(SemRelDataset(test_df,  tokenizer, MAX_LEN), 
                              batch_size=BATCH_SIZE) 
 
    model     = AfriBERTaModel(MODEL_NAME).to(DEVICE) 
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01) 
    total_steps = len(train_loader) * EPOCHS 
    scheduler = get_linear_schedule_with_warmup( 
        optimizer, 
        num_warmup_steps=int(0.1 * total_steps), 
        num_training_steps=total_steps 
    ) 
    loss_fn = torch.nn.MSELoss() 
 
    best_rho, patience_counter, best_epoch = -1, 0, 0 
 
    for epoch in range(1, EPOCHS + 1): 
        model.train() 
        total_loss = 0 
        for batch in train_loader: 
            ids   = batch['input_ids'].to(DEVICE) 
            mask  = batch['attention_mask'].to(DEVICE) 
            lbls  = batch['label'].to(DEVICE) 
            optimizer.zero_grad() 
            preds = model(ids, mask) 
            loss  = loss_fn(preds, lbls) 
            loss.backward() 
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) 
            optimizer.step() 
            scheduler.step() 
            total_loss += loss.item() 
 
        dev_rho, dev_mse, _ = evaluate(model, dev_loader) 
        avg_loss = total_loss / len(train_loader) 
        print(f"  Epoch {epoch:2d} | loss={avg_loss:.4f} | dev ρ={dev_rho:.4f} | dev MSE={dev_mse:.4f}")
 
        if dev_rho > best_rho: 
            best_rho = dev_rho 
            best_epoch = epoch 
            patience_counter = 0 
            torch.save(model.state_dict(), 
                       f'{CKPT_DIR}/afriberta_{lang_code}_best.pt') 
            print(f"           ✓ New best saved (ρ={best_rho:.4f})") 
        else: 
            patience_counter += 1 
            if patience_counter >= PATIENCE: 
                print(f"  Early stopping at epoch {epoch} (patience={PATIENCE})") 
                break 
 
    model.load_state_dict(torch.load(f'{CKPT_DIR}/afriberta_{lang_code}_best.pt')) 
    test_rho, test_mse, _ = evaluate(model, test_loader) 
    print(f"\n  Best epoch: {best_epoch} | Test ρ={test_rho:.4f} | Test MSE={test_mse:.4f}") 
    return test_rho, test_mse 
 
print("AfriBERTa functions defined.")

AfriBERTa functions defined.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) 
 
for lang_code, lang_name in [('eng','English'), ('hau','Hausa'), ('kin','Kinyarwanda')]: 
    rho, mse = train(lang_code, lang_name, tokenizer) 
 
    with open(RESULTS_LOG, 'a', newline='') as f: 
        writer = csv.DictWriter(f, fieldnames=[ 
            'experiment_id','model','model_variant','language', 
            'split','spearman_rho','mse','augmented','notes' 
        ]) 
        writer.writerow({ 
            'experiment_id': f'TL-4-{lang_code}', 
            'model': 'afriberta_finetuned', 
            'model_variant': 'castorini/afriberta_large', 
            'language': lang_name, 
            'split': 'test', 
            'spearman_rho': round(rho, 4), 
            'mse': round(mse, 4), 
            'augmented': False, 
            'notes': f'AfriBERTa fine-tuned on {lang_name} train set.' 
        }) 
    print(f"✓ {lang_name} logged: ρ={rho:.4f}, MSE={mse:.4f}") 

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


  Training AfriBERTa on English


pytorch_model.bin:   0%|          | 0.00/503M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/503M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def evaluate_zero_shot_afriberta(lang_code, lang_name): 
    print(f"\nAfriBERTa zero-shot on {lang_name}...") 
    test_df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv') 
    test_loader = DataLoader( 
        SemRelDataset(test_df, tokenizer, MAX_LEN), 
        batch_size=BATCH_SIZE 
    ) 
    model = AfriBERTaModel(MODEL_NAME).to(DEVICE) 
    model.load_state_dict(torch.load(f'{CKPT_DIR}/afriberta_eng_best.pt')) 
    rho, mse, _ = evaluate(model, test_loader) 
    print(f"  {lang_name}: ρ={rho:.4f}, MSE={mse:.4f}") 
 
    with open(RESULTS_LOG, 'a', newline='') as f: 
        writer = csv.DictWriter(f, fieldnames=[ 
            'experiment_id','model','model_variant','language', 
            'split','spearman_rho','mse','augmented','notes' 
        ]) 
        writer.writerow({ 
            'experiment_id': f'TL-3-{lang_code}', 
            'model': 'afriberta_zeroshot', 
            'model_variant': 'castorini/afriberta_large', 
            'language': lang_name, 
            'split': 'test', 
            'spearman_rho': round(rho, 4), 
            'mse': round(mse, 4), 
            'augmented': False, 
            'notes': 'AfriBERTa zero-shot: English fine-tuned checkpoint on African language.' 
        }) 
    return rho, mse 
 
# Zero-shot on all three African languages 
for lang_code, lang_name in [('afr','Afrikaans'), ('hau','Hausa'), ('kin','Kinyarwanda')]: 
    evaluate_zero_shot_afriberta(lang_code, lang_name) 

In [ ]:
# ── Always run this last cell before ending Kaggle session ── 
import pandas as pd 
 
df = pd.read_csv('/kaggle/working/results_log.csv') 
print(f"\n✓ {len(df)} results saved to results_log.csv") 
print(df[['experiment_id','language','spearman_rho','mse']].to_string(index=False)) 
 
# This file will be available in Output panel for download 
print("\n✓ Download results_log.csv from the Output panel on the right.") 